In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

def close_crm_users(db: str, bronze: str, silver: str):

    try:
        # ---------------------------
        # 1. UTC session
        # ---------------------------
        spark.conf.set("spark.sql.session.timeZone", "UTC")

        # ---------------------------
        # 2. Ensure target table
        # ---------------------------
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {db}.{silver}.CLOSE_CRM_USERS_PROCESSED (
            USER_ID STRING,
            EMAIL STRING,
            FIRST_NAME STRING,
            LAST_NAME STRING,
            MD5_HASH STRING,
            DATE_CREATED TIMESTAMP,
            DATE_UPDATED TIMESTAMP,
            INSERT_DATE TIMESTAMP,
            UPDATE_DATE TIMESTAMP
        )
        USING DELTA
        """)

        # ---------------------------
        # 3. Load bronze
        # ---------------------------
        bronze_df = spark.table(f"{db}.{bronze}.CLOSE_CRM_USERS_RAW")

        # ---------------------------
        # 4. Extract outer JSON safely (NO from_json here)
        # ---------------------------
        parsed_df = bronze_df.withColumn(
            "json_str",
            F.col("raw_data")
        ).withColumn(
            "json_object",
            F.get_json_object(F.col("json_str"), "$.JSON_OBJECT")
        ).withColumn(
            "insert_date",
            F.get_json_object(F.col("json_str"), "$.INSERT_DATE")
        )

        # ---------------------------
        # 5. Fix inner JSON (Snowflake single quotes issue)
        # ---------------------------
        parsed_df = parsed_df.withColumn(
            "json_fixed",
            F.regexp_replace(F.col("json_object"), "'", '"')
        )
        display(parsed_df)
        # ---------------------------
        # 6. Parse inner JSON (REAL DATA)
        # ---------------------------
        parsed_df = parsed_df.withColumn(
            "parsed",
            F.from_json(
                F.col("json_fixed"),
                "struct<data:array<struct<id:string,email:string,first_name:string,last_name:string,date_created:string,date_updated:timestamp>>>"
            )
        )

        # ---------------------------
        # 7. Flatten
        # ---------------------------
        exploded_df = parsed_df.select(
            F.explode("parsed.data").alias("data"),
            F.col("insert_date").alias("insert_date")
        )

        # ---------------------------
        # 8. Transform
        # ---------------------------
        transformed_df = exploded_df.select(
            F.col("data.id").alias("user_id"),
            F.col("data.email").alias("email"),
            F.col("data.first_name").alias("first_name"),
            F.col("data.last_name").alias("last_name"),
            F.col("data.date_created").alias("date_created"),
            F.col("data.date_updated").alias("date_updated"),
            F.current_timestamp().alias("update_date"),
            F.col("insert_date").cast("timestamp").alias("insert_date"),

            F.md5(F.concat_ws(
                "",
                F.coalesce(F.col("data.id"), F.lit("")),
                F.coalesce(F.col("data.email"), F.lit("")),
                F.coalesce(F.col("data.first_name"), F.lit("")),
                F.coalesce(F.col("data.last_name"), F.lit("")),
                F.coalesce(F.col("data.date_created"), F.lit("")),
                F.coalesce(F.col("data.date_updated").cast("string"), F.lit(""))
            )).alias("MD5_HASH")
        )

        # ---------------------------
        # 9. Deduplication
        # ---------------------------
        window_spec = Window.partitionBy("user_id").orderBy(F.col("insert_date").desc())

        source_df = transformed_df.withColumn(
            "rn", F.row_number().over(window_spec)
        ).filter(F.col("rn") == 1).drop("rn")

        # ---------------------------
        # 10. MERGE into Delta
        # ---------------------------
        target_table = DeltaTable.forName(
            spark, f"{db}.{silver}.CLOSE_CRM_USERS_PROCESSED"
        )

        target_table.alias("target").merge(
            source_df.alias("source"),
            "target.user_id = source.user_id AND target.email = source.email"
        ).whenMatchedUpdate(set={
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "date_created": "source.date_created",
            "date_updated": "source.date_updated",
            "update_date": "current_timestamp()",
            "MD5_HASH": "source.MD5_HASH"
        }).whenNotMatchedInsert(values={
            "user_id": "source.user_id",
            "email": "source.email",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "date_created": "source.date_created",
            "date_updated": "source.date_updated",
            "insert_date": "source.insert_date",
            "update_date": "source.update_date",
            "MD5_HASH": "source.MD5_HASH"
        }).execute()

        return "Stored Procedure Executed Successfully"

    except Exception as err:
        raise Exception(f"Error: {str(err)}")


# CALL FUNCTION
close_crm_users("customer_health", "BRONZE", "SILVER")